# Sentimiento: preparar una referencia humana de la muestra

Los 314 cambios de etiqueta no dicen cuál modelo acierta. Esta libreta prepara una evaluación sobre **60 de los 987 textos comunes** a ambos sistemas, sin nueva inferencia. Las predicciones ya guardadas se compararán con las mismas etiquetas humanas.

**Estado inicial: pendiente de anotación.** No hay accuracy ni F1 nuevos mientras falte revisar algún caso. Los resultados históricos de Robertuito permanecen separados en `evaluacion-historica-robertuito.ipynb`.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/voc").is_dir())
NB_ROOT = REPO / "reports/difusion"
if str(NB_ROOT) not in sys.path: sys.path.insert(0, str(NB_ROOT))
from apoyo_validacion import digest, export_bundle, show_texts
from apoyo_revision import verified_bundle
plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False})

## Regla fijada antes de anotar

Se asignan 60 lugares proporcionalmente por plataforma mediante restos mayores; dentro de cada fuente se ordenan IDs por SHA-256 con semilla 42. No se usa la predicción, las estrellas ni el tópico para elegir reseñas. Se conserva la unidad «registro», incluidos textos repetidos si corresponden a IDs diferentes. YouTube no aparece porque no tiene registros retenidos en la intersección.

Es una evaluación descriptiva pequeña sobre los textos comunes, sin medir los descartes del filtro. No se presenta como prueba independiente: faltan los IDs individuales del entrenamiento histórico para descartar solapamiento. Las métricas describirán exclusivamente los casos revisados, sin intervalos ni extrapolación al corpus. No se ajustarán modelos o umbrales con estas etiquetas.

In [2]:
from dataclasses import asdict
from voc.analysis.store import read_analysis
from voc_ml.experiments.operations import load_protocol
from voc.paths import data_root
from apoyo_revision import proportional_sample, sample_identity, sentiment_sheet, sentiment_metrics, blind_packet
ROOT=data_root()/'experiments/julaug-2026'
protocol=load_protocol(ROOT)
runs={s:json.loads((ROOT/f'{s}-run.json').read_text()) for s in ['base','alternative']}
frames={}
for system,run in runs.items():
    assert run['state']=='completed' and run['representation']['tracking_verified']
    saved=read_analysis(run['representation']['analysis_id'])
    labels=pd.DataFrame([asdict(x) for x in saved.sentiments])
    assert labels.status.eq('classified').all()
    frames[system]=saved.reviews[['record_id','platform','model_text']].merge(labels[['record_id','model_label']],on='record_id',validate='one_to_one')
paired=frames['base'].merge(frames['alternative'],on='record_id',suffixes=('_base','_alternative'),validate='one_to_one')
assert len(paired)==987
assert paired.model_text_base.equals(paired.model_text_alternative)
assert paired.platform_base.equals(paired.platform_alternative)
population=paired[['record_id','platform_base','model_text_base']].rename(columns={'platform_base':'platform','model_text_base':'model_text'})
sample=proportional_sample(population,60)
sample_id=sample_identity(sample,protocol['protocol_id'])
coverage=population.groupby('platform').size().rename('poblacion').to_frame().join(sample.groupby('platform').size().rename('seleccionados')).reset_index()
display(coverage)
print('Textos repetidos en la selección:',int(sample.model_text.duplicated().sum()))
print('Identidad de selección:',sample_id)

,platform,poblacion,seleccionados
0,app_store,49,3
1,google_play,876,53
2,x,62,4


Textos repetidos en la selección: 4
Identidad de selección: 3b0478cd19b3f9edbef998fe81c29b7ddba6a06bd88058d15837a314d2b48f5f


## Anotar sin consultar las predicciones

Abre `revision-manual/sentimiento-fichas.html` y edita `revision-manual/sentimiento.csv` en un editor de CSV. El paquete no incluye estrellas, tópicos ni predicciones. Valora el tono del texto completo:

- `negative`: descontento, crítica o experiencia negativa.
- `positive`: satisfacción o valoración positiva.
- `neutral`: información o consulta sin valoración afectiva reconocible.
- `uncertain`: ambigüedad, contexto insuficiente o mezcla sin predominio claro. **No equivale a neutral.**

Registra justificación, revisor y fecha en todos los casos, incluidos los inciertos. No modifiques IDs, texto ni selección. Si ya conocías una predicción, deja constancia en la justificación: el ocultamiento de esta ficha no borra exposición previa. Una sola persona no permite medir acuerdo entre anotadores.

Las anotaciones se conservan al ejecutar nuevamente. No se rellenan con propuestas automáticas.

In [3]:
review_path=NB_ROOT/'revision-manual/sentimiento.csv'
review=sentiment_sheet(review_path,sample,sample_id)
packet_path=NB_ROOT/'revision-manual/sentimiento-fichas.html'
packet_path.write_text(blind_packet(sample),encoding='utf-8')
print('Ficha sin predicciones:',packet_path)
print('Archivo para anotar:',review_path)
print('Revisados:',int(review.completa.sum()),'de',len(review))

Ficha sin predicciones: /workspace/vaults/sonar/code/repos/voc-difusion/reports/difusion/revision-manual/sentimiento-fichas.html
Archivo para anotar: /workspace/vaults/sonar/code/repos/voc-difusion/reports/difusion/revision-manual/sentimiento.csv
Revisados: 0 de 60


## Recargar la revisión y obtener resultados

Ejecuta desde aquí después de guardar el CSV. Se exige completar todos los casos antes de calcular métricas. Los `uncertain` quedan excluidos de las métricas y se reportan explícitamente; ambos modelos se evalúan sobre exactamente la misma referencia restante. Esto puede cambiar su distribución, por lo que se muestra soporte por clase.

Macro-F1 promedia las tres clases fijas. Si una clase no tiene soporte o predicciones, su métrica indefinida toma cero por convención y debe leerse junto con los soportes. No se usa accuracy como único criterio ni se interpreta una diferencia pequeña como mejora comprobada.

In [4]:
review=sentiment_sheet(review_path,sample,sample_id)
predictions=paired[['record_id','model_label_base','model_label_alternative']].rename(columns={'model_label_base':'base','model_label_alternative':'alternative'})
status,metrics,classes,matrices=sentiment_metrics(review,predictions)
print(json.dumps(status,ensure_ascii=False,indent=2))
figures={}
if not metrics.empty:
    display(metrics);display(classes)
    fig,axes=plt.subplots(1,2,figsize=(9,4))
    for ax,(system,matrix) in zip(axes,matrices.items()):
        ax.imshow(matrix,cmap='Blues',vmin=0,vmax=max(int(m.max()) for m in matrices.values()))
        ax.set(title=system,xlabel='Predicción',ylabel='Referencia humana',xticks=range(3),yticks=range(3),xticklabels=['neg','neu','pos'],yticklabels=['neg','neu','pos'])
        for (i,j),value in np.ndenumerate(matrix):ax.text(j,i,str(value),ha='center',va='center')
    fig.suptitle('Mismos textos evaluables · referencia humana de la muestra')
    fig.tight_layout();plt.show();figures['sentimiento-confusion-humana']=fig
else:
    print('Sin métricas de desempeño: la referencia humana aún no está disponible o no tiene casos evaluables.')

{
  "selected": 60,
  "reviewed": 0,
  "uncertain": 0,
  "state": "pending_human_review",
  "independent_test": false,
  "training_overlap": "unknown: individual historical training/test IDs unavailable"
}
Sin métricas de desempeño: la referencia humana aún no está disponible o no tiene casos evaluables.


In [5]:
sources={'review':{'path':str(review_path.relative_to(REPO)),'sha256':digest(review_path)},
 'review_helper':{'path':'reports/difusion/apoyo_revision.py','sha256':digest(NB_ROOT/'apoyo_revision.py')},
 'protocol':{'path':str((ROOT/'protocol.json').relative_to(REPO)),'sha256':digest(ROOT/'protocol.json')}}
for system,run in runs.items():
    path=data_root()/'analyses'/run['representation']['analysis_id']/'manifest.json'
    sources[system]={'path':str(path.relative_to(REPO)),'sha256':digest(path)}
tables={'seleccion-cobertura':coverage,'referencia-humana':review}
if not metrics.empty:tables.update({'sentimiento-metricas-humanas':metrics,'sentimiento-por-clase':classes})
exported=export_bundle(NB_ROOT/'validacion-sentimiento-muestra.ipynb',figures,tables,
 {'estado':status,'seleccion':{'sample_id':sample_id,'protocol_id':protocol['protocol_id'],'rows':len(sample),
 'rule':'platform proportional largest remainder; SHA256 record ID order; seed 42; no prediction-based selection',
 'record_ids':sample.record_id.tolist()}},sources,
 'Human sentiment reference on a fixed 60-record sample; pending labels are not model quality results')

Evidencia exportada: /workspace/vaults/sonar/code/repos/voc-difusion/reports/difusion/exports/validacion-sentimiento-muestra/20261106T231319955802Z
